# Experiment 1.0 — Sampling-rate information-preservation benchmark

## Research question

**Does reducing the sampling rate materially reduce the letter-discriminative information preserved by the encoder/reconstruction pipeline?**

This notebook is the sampling-rate gate before Experiments 1.1–1.3. It accepts **two or more dataset groups**, where every root inside one group must declare the same `sampling_rate_hz`. A group can contain Action 0 + Action 1 roots, or more compatible combination roots.

The notebook combines three complementary analyses:

### Experiment 1.0A — Native-rate reconstruction benchmark

`reconstructed acceleration -> CNN-S/M/L trained from scratch -> test classification + frozen-embedding probes`

For every sampling rate and every CNN size, the model is trained from scratch on the reconstructed 3-axis acceleration. The primary result is test **balanced accuracy**. Frozen embeddings are additionally evaluated using a **linear probe** and **kNN**.

This is the most direct practical question: *if the reconstructed signal is consumed at its native sample rate, does downstream classification performance degrade when the rate is reduced?*

### Experiment 1.0B — Common-time-grid control

`native reconstruction -> deterministic interpolation to one common evaluation rate -> same CNN-S/M/L -> same metrics`

The CNN kernels are defined in **samples**, so the same CNN has a different physical receptive field at 64 Hz and 200 Hz. For example, a 27-sample receptive field spans about 422 ms at 64 Hz but 135 ms at 200 Hz. Therefore the native-rate comparison alone cannot cleanly separate signal-information preservation from a change in the decoder's physical temporal context.

The common-grid control maps every valid reconstructed trajectory onto the **same evaluation time grid** before training. By default the highest native rate is used, so lower-rate signals are only interpolated upward; interpolation does not restore high-frequency information that was lost before reconstruction. After this conversion, a CNN kernel spans the same physical time for every dataset.

### Experiment 1.0C — Direct-event 10-bin control

`weighted events -> 10 relative-time bins -> per-channel weighted sum -> flatten -> linear classifier`

This reproduces the core idea of Experiment 1.3 as an auxiliary encoder-domain check. It asks whether the **direct weighted event representation** still retains coarse temporal-position information after reducing the sample rate. This control is run only when the event schema is comparable across all dataset groups.

---

## Primary interpretation

The main claim should be based on converging evidence, not one number:

1. **CNN-L test balanced accuracy** — primary downstream metric.
2. **CNN-L frozen-embedding linear-probe balanced accuracy** — primary representation-quality support.
3. **CNN-L kNN balanced accuracy** — local neighborhood structure in the learned embedding.
4. **Macro-F1** — class-balanced error structure.
5. **CNN-S/M** — capacity controls, not the primary information estimator.
6. **1.0B common-grid CNN-L** — the key control for physical receptive-field differences.
7. **1.0C event 10-bin probe** — auxiliary evidence that the direct event code remains informative.

Accuracy is retained for completeness but is not the main metric when class counts differ.

## Critical controls

- The comparison uses the **intersection of canonical sample IDs across all sampling rates**.
- The exact same user-disjoint train/validation/test split is reused everywhere.
- Five paired master seeds control model initialization, DataLoader order, linear-probe randomness, and other stochastic operations.
- For a fixed seed and CNN variant, model initialization and sample order are paired across sampling rates.
- Normalization is fitted **only on training valid samples**, separately for each sampling-rate representation.
- Test data never selects an epoch, kNN `k`, or linear-probe checkpoint.
- Reconstruction comparisons should ideally differ **only in sampling rate**. Encoder/event schema mismatches are audited and raise by default.
- The common-grid control never downsamples unless explicitly changed; by default it uses the highest available native rate.

A result such as `64 Hz BA ~= 200 Hz BA` supports **no observed material degradation**. It does not prove mathematical equivalence. The notebook therefore also reports paired deltas relative to the highest-rate reference and a configurable practical degradation margin.


In [3]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import hashlib
import json
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the writingRing repository root from the current working directory"
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from snn.accel_reconstruction_eval.datasets import load_acceleration_data
from snn.accel_reconstruction_eval.model import build_probe_model
from snn.accel_reconstruction_eval.embedding import (
    extract_embeddings,
    standardize_feature_splits,
)
from snn.accel_reconstruction_eval.evaluation import (
    LinearProbeConfig,
    evaluate_linear_probe,
    evaluate_retrieval,
    train_linear_probe,
)

print("Repository root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Repository root: /home/ted/project/writingring-viz
PyTorch: 2.11.0+cu130
CUDA available: True


## 1. User configuration

Edit this cell for routine runs.

`DATASET_GROUPS` is a mapping from a human-readable name to one or more compatible combination roots. The notebook reads the true sample rate from producer metadata; the dictionary key is only a label.

For a clean sample-rate study, the groups should use the **same labels, same subjects, same segmentation convention, same encoder/wavelet configuration, same polarity convention, and same reconstruction algorithm**, with sampling rate as the intended changing factor.


In [4]:
# -----------------------------------------------------------------------------
# USER CONFIGURATION
# -----------------------------------------------------------------------------
# Replace these example paths with the actual matched datasets to compare.
# Each list can contain Action 0 + Action 1, or more compatible roots.
DATASET_GROUPS = {
    "sr_200": [
        REPO_ROOT / "outputs/action0_wavelets_0e5_1_2_4_8",
        REPO_ROOT / "outputs/action1_wavelets_0e5_1_2_4_8",
    ],
    "sr_64": [
        REPO_ROOT / "outputs/action0_wavelets_0e5_1_2_4_8_sr_64",
        REPO_ROOT / "outputs/action1_wavelets_0e5_1_2_4_8_sr_64",
    ],
}

# Five paired master seeds used throughout the notebook.
SEEDS = (11, 23, 37, 53, 71)
assert len(SEEDS) == 5 and len(set(SEEDS)) == 5

# One user-disjoint cohort split shared by every sampling rate, model size, and seed.
SPLIT_SEED = 12345
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15

# None keeps every common label. Otherwise only these labels are retained.
INCLUDED_LABELS: tuple[str, ...] | None = (
    "A", "B", "C", "D", "E", "X", "G", "H", "I", "J", "K", "L"
)

CNN_VARIANTS = ("cnn_s", "cnn_m", "cnn_l")

# Shared CNN training budget.
BATCH_SIZE = 128
NUM_WORKERS = 0
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 10
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.0
USE_CLASS_WEIGHTS = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Repository-consistent frozen-embedding probes.
K_VALUES = (1, 3, 5, 10, 20, 50)
KNN_K_CANDIDATES = (1, 3, 5, 7, 10, 15, 20)
LINEAR_PROBE_CONFIG = LinearProbeConfig(
    epochs=200,
    learning_rate=1e-2,
    weight_decay=1e-4,
    batch_size=512,
    patience=30,
    seed_offset=1,
)

# Experiment 1.0B: common-time-grid control.
RUN_COMMON_GRID_CONTROL = True
# None -> automatically use the highest native sampling rate found in DATASET_GROUPS.
COMMON_GRID_RATE_HZ: float | None = None
# Run all three by default. For a faster confirmation run, set ("cnn_l",).
COMMON_GRID_VARIANTS = CNN_VARIANTS

# Experiment 1.0C: direct-event coarse temporal-position control.
RUN_EVENT_10BIN_CONTROL = True
EVENT_N_BINS = 10
# By default a schema mismatch skips 1.0C. Do not force unlike event encodings into
# one sampling-rate conclusion.
SKIP_EVENT_CONTROL_ON_SCHEMA_MISMATCH = True

# To isolate sampling rate, mismatched encoder/event schemas are treated as a
# methodological error by default. Set True only for exploratory/non-causal comparisons.
ALLOW_ENCODER_SCHEMA_MISMATCH = False

# Practical, not statistical, degradation margin for paired delta summaries.
# Example: -0.03 means a loss smaller than 3 BA points is considered small in practice.
PRACTICAL_BA_MARGIN = 0.03

SAVE_CHECKPOINTS = False
RESULTS_DIR = REPO_ROOT / "notebooks/artifacts/experiment_1_0_sampling_rate_information_preservation"

print("Dataset groups:")
for name, roots in DATASET_GROUPS.items():
    print(f"[{name}]")
    for root in roots:
        print(" -", root)
print("Seeds:", SEEDS)
print("CNN variants:", CNN_VARIANTS)
print("Device:", DEVICE)


Dataset groups:
[sr_200]
 - /home/ted/project/writingring-viz/outputs/action0_wavelets_0e5_1_2_4_8
 - /home/ted/project/writingring-viz/outputs/action1_wavelets_0e5_1_2_4_8
[sr_64]
 - /home/ted/project/writingring-viz/outputs/action0_wavelets_0e5_1_2_4_8_sr_64
 - /home/ted/project/writingring-viz/outputs/action1_wavelets_0e5_1_2_4_8_sr_64
Seeds: (11, 23, 37, 53, 71)
CNN variants: ('cnn_s', 'cnn_m', 'cnn_l')
Device: cuda


## 2. Reproducibility helpers

The split seed is intentionally separate from the five model seeds. A SHA-256-derived sub-seed makes stochastic operations stable across Python processes. For a fixed master seed and CNN variant, **group name is intentionally omitted from the model-initialization and loader-order seeds**, so corresponding runs at different sample rates start from identical weights and see common samples in the same order.


In [5]:
def derive_seed(master_seed: int, *parts: object) -> int:
    text = "|".join([str(master_seed), *(str(p) for p in parts)])
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little", signed=False)


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def metric_dict(y_true, y_pred) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


## 3. Load every dataset group and audit metadata

The repository loader is used independently for each sampling-rate group. Within a group it validates the combination roots and requires reconstruction files.

This cell records:

- actual `sampling_rate_hz`,
- number of samples/users/classes,
- target padded length,
- feature schema,
- channel count,
- event-channel identity when available.

The cross-rate comparison should ideally differ only in sampling rate. A mismatched encoder schema can change reconstruction quality independently of sample rate, so the notebook raises by default.


In [6]:
def read_group_sampling_rate_hz(data) -> float:
    rates_by_root = {
        str(root): float(metadata.sampling_rate_hz)
        for root, metadata in zip(data.padded_roots, data.producer_metadatas, strict=True)
    }
    rates = sorted(set(rates_by_root.values()))
    if len(rates) != 1:
        raise ValueError(
            "Roots inside one DATASET_GROUP must share sampling_rate_hz; "
            f"got {rates_by_root}"
        )
    rate = rates[0]
    if not np.isfinite(rate) or rate <= 0:
        raise ValueError(f"Invalid sampling rate: {rate}")
    return rate


def metadata_signature(data) -> dict[str, object]:
    """Build a conservative encoder/event signature for cross-rate auditing."""
    schemas = tuple(sorted({str(getattr(m, "feature_schema", None)) for m in data.producer_metadatas}))
    channel_counts = tuple(sorted({int(getattr(m, "channel_count", -1)) for m in data.producer_metadatas}))

    names_per_root = []
    for m in data.producer_metadatas:
        names = getattr(m, "channel_names", None)
        if names is None:
            raw = getattr(m, "raw", None)
            if isinstance(raw, dict):
                names = raw.get("channel_names")
        names_per_root.append(tuple(map(str, names)) if names is not None else ())

    # The padded SpikeIMU convention is event channels followed by accel+gyro (6 channels).
    package_channel_counts = tuple(sorted({int(p.padded_spike_imu.shape[-1]) for p in data.packages}))
    if len(package_channel_counts) != 1:
        raise ValueError(f"Inconsistent padded_spike_imu channel counts: {package_channel_counts}")
    total_channels = package_channel_counts[0]
    inferred_event_count = total_channels - 6
    if inferred_event_count <= 0:
        raise ValueError(f"Cannot infer event channels from total channel count={total_channels}")

    event_names = ()
    if names_per_root and all(names_per_root) and len(set(names_per_root)) == 1:
        names = names_per_root[0]
        if len(names) >= inferred_event_count:
            event_names = tuple(names[:inferred_event_count])

    return {
        "feature_schema": schemas,
        "metadata_channel_count": channel_counts,
        "padded_channel_count": total_channels,
        "event_count": inferred_event_count,
        "event_names": event_names,
    }


def build_manifest(data, group_name: str, sampling_rate_hz: float) -> pd.DataFrame:
    """Build a cross-rate manifest using source_segment_index as stable identity.

    The repository sample_id uses output_segment_index, which can shift if one
    sampling-rate dataset skips a different overflow segment. For cross-rate
    matching we therefore key samples by the original source_segment_index.
    """
    df = data.sample_manifest.copy()
    if INCLUDED_LABELS is not None:
        df = df[df["label"].astype(str).isin(set(map(str, INCLUDED_LABELS)))].copy()
    if df.empty:
        raise ValueError(f"{group_name}: no samples remain after label filtering")

    required = {
        "user", "action", "stem", "segment_index", "source_segment_index",
        "package_index", "label", "valid_length",
    }
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{group_name}: repository sample manifest is missing {sorted(missing)}")

    df["repo_sample_id"] = df["sample_id"].astype(str)
    df["sample_id"] = [
        f"{u}/action_{a}/{stem}/source_segment_{int(src):06d}"
        for u, a, stem, src in zip(
            df.user.astype(str),
            df.action.astype(str),
            df.stem.astype(str),
            df.source_segment_index.astype(int),
            strict=True,
        )
    ]
    df["group"] = group_name
    df["sampling_rate_hz"] = float(sampling_rate_hz)
    df["label"] = df["label"].astype(str)
    df["valid_length"] = df["valid_length"].astype(int)
    df["package_index"] = df["package_index"].astype(int)
    df["segment_index"] = df["segment_index"].astype(int)

    if df["sample_id"].duplicated().any():
        dup = df.loc[df["sample_id"].duplicated(), "sample_id"].head().tolist()
        raise ValueError(f"{group_name}: duplicate cross-rate sample IDs, e.g. {dup}")
    return df.sort_values("sample_id").reset_index(drop=True)


loaded_groups: dict[str, object] = {}
group_manifests: dict[str, pd.DataFrame] = {}
group_rates: dict[str, float] = {}
group_signatures: dict[str, dict[str, object]] = {}
audit_rows = []

for group_name, roots in DATASET_GROUPS.items():
    data = load_acceleration_data(
        roots,
        repository_root=REPO_ROOT,
        require_reconstruction=True,
    )
    rate = read_group_sampling_rate_hz(data)
    signature = metadata_signature(data)
    manifest = build_manifest(data, group_name, rate)

    loaded_groups[group_name] = data
    group_rates[group_name] = rate
    group_signatures[group_name] = signature
    group_manifests[group_name] = manifest

    target_lengths = sorted({int(p.padded_spike_imu.shape[1]) for p in data.packages})
    recon_lengths = sorted({int(p.reconstructed_acceleration.shape[1]) for p in data.packages if p.reconstructed_acceleration is not None})
    audit_rows.append({
        "group": group_name,
        "sampling_rate_hz": rate,
        "samples_before_common_intersection": len(manifest),
        "users": manifest.user.nunique(),
        "classes": manifest.label.nunique(),
        "spike_target_lengths": target_lengths,
        "reconstruction_target_lengths": recon_lengths,
        "feature_schema": signature["feature_schema"],
        "event_count": signature["event_count"],
    })

if len(loaded_groups) < 2:
    raise ValueError("Experiment 1.0 requires at least two sampling-rate groups")

metadata_audit = pd.DataFrame(audit_rows).sort_values("sampling_rate_hz", ascending=False)
display(metadata_audit)

# Conservative cross-rate encoder audit. Sampling rate itself is intentionally not part
# of this signature.
encoder_keys = ("feature_schema", "event_count", "event_names")
reference_group = next(iter(group_signatures))
reference_signature = tuple(group_signatures[reference_group][k] for k in encoder_keys)
mismatched_groups = []
for group_name, signature in group_signatures.items():
    candidate = tuple(signature[k] for k in encoder_keys)
    if candidate != reference_signature:
        mismatched_groups.append(group_name)

if mismatched_groups:
    msg = (
        "Encoder/event schema differs across sampling-rate groups. This confounds a pure "
        f"sample-rate causal comparison. Reference={reference_group}; mismatched={mismatched_groups}."
    )
    if ALLOW_ENCODER_SCHEMA_MISMATCH:
        warnings.warn(msg)
    else:
        raise ValueError(msg + " Set ALLOW_ENCODER_SCHEMA_MISMATCH=True only for exploratory runs.")


Action0DatasetError: ROOT resolves to multiple padded datasets; select one exact root:
  - /home/ted/project/writingring-viz/outputs/action0_wavelets_0e5_1_2_4_8/low-pass/aligned-board-events/segmentation_padded
  - /home/ted/project/writingring-viz/outputs/action0_wavelets_0e5_1_2_4_8/low-pass/aligned-board-events/segmentation_padded_polarity_split

## 4. Construct one common cohort across all sampling rates

A sampling-rate comparison is not fair if one dataset contains different gestures from another. The notebook therefore computes the intersection of `sample_id` across all groups and then verifies that the corresponding labels agree.

Only this common cohort is used in Experiments 1.0A, 1.0B, and 1.0C.


In [ ]:
common_ids = None
for manifest in group_manifests.values():
    ids = set(manifest.sample_id.tolist())
    common_ids = ids if common_ids is None else common_ids.intersection(ids)
common_ids = sorted(common_ids or [])
if not common_ids:
    raise ValueError("No canonical sample IDs are shared by all sampling-rate groups")

common_manifests: dict[str, pd.DataFrame] = {}
cohort_rows = []
for group_name, manifest in group_manifests.items():
    common = manifest[manifest.sample_id.isin(common_ids)].copy().sort_values("sample_id").reset_index(drop=True)
    common_manifests[group_name] = common
    cohort_rows.append({
        "group": group_name,
        "sampling_rate_hz": group_rates[group_name],
        "original_samples": len(manifest),
        "common_samples": len(common),
        "removed_by_intersection": len(manifest) - len(common),
        "users": common.user.nunique(),
        "classes": common.label.nunique(),
    })

# Label identity check across rates.
ref_group = max(group_rates, key=group_rates.get)
ref_labels = common_manifests[ref_group].set_index("sample_id")["label"]
for group_name, common in common_manifests.items():
    labels = common.set_index("sample_id")["label"].reindex(ref_labels.index)
    mismatch = labels != ref_labels
    if mismatch.any():
        bad = ref_labels.index[mismatch][:10].tolist()
        raise ValueError(f"Label mismatch across sampling rates for sample IDs: {bad}")

# Stable class mapping shared by every group.
if INCLUDED_LABELS is None:
    labels_ordered = sorted(ref_labels.unique().tolist())
else:
    labels_present = set(ref_labels.unique().tolist())
    labels_ordered = [str(v) for v in INCLUDED_LABELS if str(v) in labels_present]
CLASS_TO_IDX = {label: i for i, label in enumerate(labels_ordered)}
IDX_TO_CLASS = {i: label for label, i in CLASS_TO_IDX.items()}

for group_name in common_manifests:
    common_manifests[group_name]["label_idx"] = (
        common_manifests[group_name]["label"].map(CLASS_TO_IDX).astype(int)
    )

cohort_audit = pd.DataFrame(cohort_rows).sort_values("sampling_rate_hz", ascending=False)
display(cohort_audit)
print("Common sample count:", len(common_ids))
print("Common classes:", labels_ordered)


## 5. Build one fixed user-disjoint split

The split is created from the common cohort once. Every sampling rate receives the same train/validation/test users and the same sample IDs in each split. The cell asserts that every class is represented in all three splits; if that assertion fails, choose another `SPLIT_SEED` rather than silently changing cohorts per rate.


In [ ]:
def make_user_split(reference_manifest: pd.DataFrame, split_seed: int) -> dict[str, str]:
    users = np.array(sorted(reference_manifest.user.unique().tolist()), dtype=object)
    if len(users) < 3:
        raise ValueError("Need at least three users for a user-disjoint split")

    rng = np.random.default_rng(derive_seed(split_seed, "user_split"))
    rng.shuffle(users)
    n = len(users)
    n_train = max(1, int(np.floor(TRAIN_FRACTION * n)))
    n_val = max(1, int(np.floor(VAL_FRACTION * n)))
    if n_train + n_val >= n:
        n_train = n - 2
        n_val = 1

    train_users = set(users[:n_train].tolist())
    val_users = set(users[n_train:n_train + n_val].tolist())
    test_users = set(users[n_train + n_val:].tolist())

    assignment = {}
    for u in train_users:
        assignment[str(u)] = "train"
    for u in val_users:
        assignment[str(u)] = "val"
    for u in test_users:
        assignment[str(u)] = "test"
    return assignment


user_to_split = make_user_split(common_manifests[ref_group], SPLIT_SEED)
for group_name in common_manifests:
    df = common_manifests[group_name].copy()
    df["split"] = df.user.map(user_to_split)
    if df.split.isna().any():
        raise AssertionError(f"{group_name}: at least one common user has no split assignment")
    common_manifests[group_name] = df

split_review = []
for split in ("train", "val", "test"):
    part = common_manifests[ref_group][common_manifests[ref_group].split == split]
    split_review.append({
        "split": split,
        "users": part.user.nunique(),
        "samples": len(part),
        "classes_present": part.label.nunique(),
        "class_count_expected": len(CLASS_TO_IDX),
    })
split_review = pd.DataFrame(split_review)
display(split_review)

if not (split_review.classes_present == len(CLASS_TO_IDX)).all():
    raise ValueError(
        "At least one split is missing a selected class. Change SPLIT_SEED or label selection."
    )

# Explicit cross-group split identity check.
ref_split_map = common_manifests[ref_group].set_index("sample_id")["split"]
for group_name, df in common_manifests.items():
    other = df.set_index("sample_id")["split"].reindex(ref_split_map.index)
    if not other.equals(ref_split_map):
        raise AssertionError(f"Split mismatch between {ref_group} and {group_name}")


## 6. CNN physical receptive-field audit

CNN-S/M/L are reused from `snn.accel_reconstruction_eval.model.build_probe_model`; no duplicate model definitions are kept in this notebook.

The table below converts each architecture's receptive field from samples to milliseconds at every native sample rate. This is the reason Experiment 1.0B is required when making an information-preservation claim.


In [ ]:
def receptive_field_samples_from_architecture(architecture: dict[str, object]) -> int:
    rf = 1
    jump = 1
    for block in architecture["blocks"]:
        k = int(block["kernel_size"])
        s = int(block["stride"])
        rf = rf + (k - 1) * jump
        jump *= s
    return int(rf)


rf_rows = []
for variant in CNN_VARIANTS:
    probe = build_probe_model(variant, num_classes=len(CLASS_TO_IDX))
    architecture = probe.architecture_config()
    rf_samples = receptive_field_samples_from_architecture(architecture)
    for group_name, rate in group_rates.items():
        rf_rows.append({
            "model": variant,
            "sampling_rate_hz": rate,
            "receptive_field_samples": rf_samples,
            "receptive_field_ms": 1000.0 * rf_samples / rate,
            "embedding_dim": int(architecture["embedding_dim"]),
        })
rf_table = pd.DataFrame(rf_rows).sort_values(["model", "sampling_rate_hz"], ascending=[True, False])
display(rf_table)


## 7. Reconstruction dataset views: native rate and common time grid

### Native view

The reconstructed array is consumed at the dataset's own sample rate. Only the valid prefix is normalized; right padding remains zero.

### Common-grid view

The valid reconstructed prefix is interpolated to `COMMON_GRID_RATE_HZ`. By default this is the highest native rate, which avoids introducing an additional notebook-side low-pass/downsampling operation. A lower-rate reconstruction mapped upward still contains only the information that survived the lower-rate pipeline.

Normalization is fitted **after** the view transform using training valid samples only.


In [ ]:
def resampled_valid_length(valid_length: int, src_rate_hz: float, dst_rate_hz: float) -> int:
    valid_length = int(valid_length)
    if valid_length <= 0:
        raise ValueError("valid_length must be positive")
    if valid_length == 1:
        return 1
    duration_s = (valid_length - 1) / float(src_rate_hz)
    return int(round(duration_s * float(dst_rate_hz))) + 1


def interpolate_valid_prefix(values: np.ndarray, src_rate_hz: float, dst_rate_hz: float) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    if values.ndim != 2 or values.shape[1] != 3:
        raise ValueError(f"Expected [T,3] reconstruction prefix, got {values.shape}")
    if len(values) == 1 or np.isclose(src_rate_hz, dst_rate_hz):
        return np.array(values, copy=True)

    target_length = resampled_valid_length(len(values), src_rate_hz, dst_rate_hz)
    src_t = np.arange(len(values), dtype=np.float64) / float(src_rate_hz)
    dst_t = np.arange(target_length, dtype=np.float64) / float(dst_rate_hz)
    # Floating-point rounding can make the last destination point infinitesimally later.
    dst_t = np.minimum(dst_t, src_t[-1])
    out = np.stack(
        [np.interp(dst_t, src_t, values[:, c]) for c in range(values.shape[1])],
        axis=1,
    )
    return out.astype(np.float32, copy=False)


if COMMON_GRID_RATE_HZ is None:
    COMMON_GRID_RATE_HZ = float(max(group_rates.values()))
COMMON_GRID_RATE_HZ = float(COMMON_GRID_RATE_HZ)
if COMMON_GRID_RATE_HZ + 1e-9 < max(group_rates.values()):
    raise ValueError(
        "COMMON_GRID_RATE_HZ is below at least one native rate. This notebook intentionally "
        "avoids notebook-side downsampling without an explicit anti-aliasing design."
    )
print("Common evaluation grid:", COMMON_GRID_RATE_HZ, "Hz")

# One common padded length for the common-grid view across every group.
common_grid_pad_length = 1
for group_name, df in common_manifests.items():
    src_rate = group_rates[group_name]
    max_len = max(
        resampled_valid_length(int(L), src_rate, COMMON_GRID_RATE_HZ)
        for L in df.valid_length.tolist()
    )
    common_grid_pad_length = max(common_grid_pad_length, max_len)
print("Common-grid padded length:", common_grid_pad_length, "samples")


def reconstruction_valid_array(data, row, view: str, src_rate_hz: float) -> np.ndarray:
    package = data.packages[int(row.package_index)]
    i = int(row.segment_index)
    L = int(row.valid_length)
    if package.reconstructed_acceleration is None:
        raise RuntimeError("Reconstruction was required but is missing")
    valid = np.asarray(package.reconstructed_acceleration[i, :L], dtype=np.float32)
    if view == "native":
        return np.array(valid, copy=True)
    if view == "common_grid":
        return interpolate_valid_prefix(valid, src_rate_hz, COMMON_GRID_RATE_HZ)
    raise ValueError(f"Unknown reconstruction view: {view}")


def fit_reconstruction_normalization(data, train_df: pd.DataFrame, view: str, src_rate_hz: float):
    total = np.zeros(3, dtype=np.float64)
    total_sq = np.zeros(3, dtype=np.float64)
    count = 0
    for row in train_df.itertuples(index=False):
        arr = reconstruction_valid_array(data, row, view, src_rate_hz).astype(np.float64)
        total += arr.sum(axis=0)
        total_sq += np.square(arr).sum(axis=0)
        count += len(arr)
    if count <= 0:
        raise ValueError("No training valid samples available for normalization")
    mean = total / count
    var = np.maximum(total_sq / count - mean**2, 1e-12)
    return mean.astype(np.float32), np.sqrt(var).astype(np.float32)


class ReconstructionViewDataset(Dataset):
    def __init__(self, data, df, src_rate_hz, view, channel_mean, channel_std):
        self.data = data
        self.df = df.reset_index(drop=True)
        self.src_rate_hz = float(src_rate_hz)
        self.view = str(view)
        self.channel_mean = np.asarray(channel_mean, dtype=np.float32)
        self.channel_std = np.asarray(channel_std, dtype=np.float32)

        if self.view == "native":
            lengths = {
                int(p.reconstructed_acceleration.shape[1])
                for p in self.data.packages
                if p.reconstructed_acceleration is not None
            }
            if len(lengths) != 1:
                raise ValueError(f"Native reconstruction padded lengths are inconsistent: {lengths}")
            self.pad_length = int(next(iter(lengths)))
        elif self.view == "common_grid":
            self.pad_length = int(common_grid_pad_length)
        else:
            raise ValueError(self.view)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        valid = reconstruction_valid_array(self.data, row, self.view, self.src_rate_hz)
        L = len(valid)
        if L > self.pad_length:
            raise ValueError(f"Valid length {L} exceeds padded length {self.pad_length}")

        x = np.zeros((self.pad_length, 3), dtype=np.float32)
        x[:L] = (valid - self.channel_mean) / self.channel_std
        valid_mask = np.arange(self.pad_length) < L
        return {
            "x": torch.from_numpy(x.T.copy()),
            "label": torch.tensor(int(row.label_idx), dtype=torch.long),
            "valid_mask": torch.from_numpy(valid_mask),
            "sample_id": str(row.sample_id),
        }


## 8. Precompute train-only normalization for every group/view

The normalization parameters are deterministic and do not depend on the model seed, so they are fitted once and reused across CNN-S/M/L and the five master seeds.


In [ ]:
normalization_cache: dict[tuple[str, str], tuple[np.ndarray, np.ndarray]] = {}
norm_rows = []
views_to_prepare = ["native"] + (["common_grid"] if RUN_COMMON_GRID_CONTROL else [])

for group_name, data in loaded_groups.items():
    df = common_manifests[group_name]
    train_df = df[df.split == "train"]
    for view in views_to_prepare:
        mean, std = fit_reconstruction_normalization(
            data,
            train_df,
            view=view,
            src_rate_hz=group_rates[group_name],
        )
        normalization_cache[(group_name, view)] = (mean, std)
        norm_rows.append({
            "group": group_name,
            "sampling_rate_hz": group_rates[group_name],
            "view": view,
            "mean_x": mean[0], "mean_y": mean[1], "mean_z": mean[2],
            "std_x": std[0], "std_y": std[1], "std_z": std[2],
        })
normalization_table = pd.DataFrame(norm_rows)
display(normalization_table)


## 9. DataLoader, training, and frozen-embedding evaluation helpers

For each `(view, CNN variant, master seed)`, the model-initialization seed and sample-order seed are shared across sample rates. Each sample rate still has its own train-fitted normalization.

Model selection uses **validation balanced accuracy**. The best checkpoint is restored before test evaluation. Frozen embeddings then use:

- the repository linear-probe protocol with train-only feature standardization,
- the repository kNN protocol with validation selection of `k`.


In [ ]:
def make_loaders(group_name: str, view: str, master_seed: int, variant: str):
    data = loaded_groups[group_name]
    df = common_manifests[group_name]
    mean, std = normalization_cache[(group_name, view)]
    src_rate = group_rates[group_name]

    datasets = {}
    for split_name in ("train", "val", "test"):
        split_df = df[df.split == split_name].sort_values("sample_id")
        datasets[split_name] = ReconstructionViewDataset(
            data, split_df, src_rate, view, mean, std
        )
    datasets["train_eval"] = ReconstructionViewDataset(
        data, df[df.split == "train"].sort_values("sample_id"), src_rate, view, mean, std
    )

    loaders = {}
    for split_name, dataset in datasets.items():
        # group_name intentionally omitted: same sample permutation across rates.
        g = torch.Generator().manual_seed(
            derive_seed(master_seed, view, variant, split_name, "loader")
        )
        loaders[split_name] = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=(split_name == "train"),
            num_workers=NUM_WORKERS,
            generator=g,
            worker_init_fn=worker_init_fn if NUM_WORKERS > 0 else None,
        )
    return loaders


@torch.inference_mode()
def evaluate_cnn(model, loader):
    model.eval()
    y_true, y_pred, sample_ids = [], [], []
    for batch in loader:
        logits, _ = model(
            batch["x"].to(DEVICE, dtype=torch.float32),
            valid_mask=batch["valid_mask"].to(DEVICE, dtype=torch.bool),
        )
        y_true.extend(batch["label"].numpy().tolist())
        y_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        sample_ids.extend(map(str, batch["sample_id"]))
    return metric_dict(y_true, y_pred), np.asarray(y_true), np.asarray(y_pred), np.asarray(sample_ids)


def fit_cnn(model, train_loader, val_loader, training_seed: int):
    seed_everything(training_seed)
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_val_ba = -np.inf
    best_epoch = -1
    no_improvement = 0
    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_n = 0
        for batch in train_loader:
            x = batch["x"].to(DEVICE, dtype=torch.float32)
            y = batch["label"].to(DEVICE, dtype=torch.long)
            mask = batch["valid_mask"].to(DEVICE, dtype=torch.bool)

            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(x, valid_mask=mask)
            loss = criterion(logits, y)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite CNN training loss")
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach()) * len(y)
            train_n += len(y)

        val_metrics, _, _, _ = evaluate_cnn(model, val_loader)
        val_ba = float(val_metrics["balanced_accuracy"])
        history_rows.append({
            "epoch": epoch,
            "train_loss": train_loss_sum / max(train_n, 1),
            "val_balanced_accuracy": val_ba,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        })

        if val_ba > best_val_ba + 1e-12:
            best_val_ba = val_ba
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improvement = 0
        else:
            no_improvement += 1
            if no_improvement >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is None:
        raise RuntimeError("CNN training never produced a best checkpoint")
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, pd.DataFrame(history_rows), int(best_epoch), float(best_val_ba)


def frozen_embedding_metrics(model, loaders, master_seed: int, view: str, variant: str):
    train_bundle = extract_embeddings(model, loaders["train_eval"], device=DEVICE)
    val_bundle = extract_embeddings(model, loaders["val"], device=DEVICE)
    test_bundle = extract_embeddings(model, loaders["test"], device=DEVICE)

    # kNN uses the repository's cosine-normalized z embeddings and selects k on validation.
    retrieval = evaluate_retrieval(
        train_bundle=train_bundle,
        val_bundle=val_bundle,
        test_bundle=test_bundle,
        k_values=K_VALUES,
        knn_k_candidates=KNN_K_CANDIDATES,
        batch_size=128,
    )

    # Linear probe follows the repository protocol: standardize h using train statistics only.
    train_x, val_x, test_x, _, _ = standardize_feature_splits(
        train_bundle.h, val_bundle.h, test_bundle.h
    )
    probe_seed = derive_seed(master_seed, view, variant, "linear_probe")
    probe, probe_history, probe_best_epoch, probe_best_val = train_linear_probe(
        train_x,
        train_bundle.y,
        val_x,
        val_bundle.y,
        num_classes=len(CLASS_TO_IDX),
        device=DEVICE,
        config=LINEAR_PROBE_CONFIG,
        random_seed=probe_seed,
    )
    lp_metrics = evaluate_linear_probe(
        probe,
        test_x,
        test_bundle.y,
        device=DEVICE,
        batch_size=LINEAR_PROBE_CONFIG.batch_size,
    )

    return {
        "linear_probe_accuracy": float(lp_metrics["accuracy"]),
        "linear_probe_balanced_accuracy": float(lp_metrics["balanced_accuracy"]),
        "linear_probe_macro_f1": float(lp_metrics["macro_f1"]),
        "linear_probe_best_epoch": int(probe_best_epoch),
        "linear_probe_best_val_balanced_accuracy": float(probe_best_val),
        "knn_accuracy": float(retrieval["knn_metrics"]["accuracy"]),
        "knn_balanced_accuracy": float(retrieval["knn_metrics"]["balanced_accuracy"]),
        "knn_macro_f1": float(retrieval["knn_metrics"]["macro_f1"]),
        "knn_selected_k": int(retrieval["selected_knn_k"]),
    }, probe_history


# Experiment 1.0A — Native-rate reconstruction benchmark

## Protocol

For every sampling rate, CNN-S/M/L, and master seed:

1. Use exactly the same common train/validation/test sample IDs.
2. Fit normalization from that sampling rate's reconstruction **training valid samples only**.
3. Initialize the selected CNN from scratch using a seed paired across rates.
4. Train on reconstructed acceleration at its **native rate**.
5. Select the best epoch by validation balanced accuracy.
6. Evaluate end-to-end CNN Accuracy / Balanced Accuracy / Macro-F1 on test.
7. Freeze the best CNN embedding.
8. Evaluate linear-probe and kNN classification on the frozen embedding.

CNN-L is the primary decoder. CNN-S and CNN-M are capacity controls.


In [ ]:
def run_reconstruction_sweep(view: str, variants: tuple[str, ...]):
    rows = []
    histories = []
    predictions = []
    probe_histories = []

    for master_seed in SEEDS:
        print(f"\n=== {view} | master seed {master_seed} ===")
        for variant in variants:
            for group_name in sorted(group_rates, key=group_rates.get, reverse=True):
                rate = group_rates[group_name]
                print(f"  {variant} | {group_name} | {rate:g} Hz")
                loaders = make_loaders(group_name, view, master_seed, variant)

                # group_name intentionally omitted for paired initialization across rates.
                init_seed = derive_seed(master_seed, view, variant, "model_init")
                seed_everything(init_seed)
                model = build_probe_model(variant, num_classes=len(CLASS_TO_IDX))

                training_seed = derive_seed(master_seed, view, variant, "training")
                model, history, best_epoch, best_val_ba = fit_cnn(
                    model, loaders["train"], loaders["val"], training_seed
                )

                train_metrics, _, _, _ = evaluate_cnn(model, loaders["train_eval"])
                val_metrics, _, _, _ = evaluate_cnn(model, loaders["val"])
                test_metrics, y_true, y_pred, sample_ids = evaluate_cnn(model, loaders["test"])
                representation_metrics, lp_history = frozen_embedding_metrics(
                    model, loaders, master_seed, view, variant
                )

                row = {
                    "view": view,
                    "group": group_name,
                    "sampling_rate_hz": rate,
                    "variant": variant,
                    "seed": master_seed,
                    "best_epoch": best_epoch,
                    "best_val_balanced_accuracy": best_val_ba,
                    "train_accuracy": train_metrics["accuracy"],
                    "train_balanced_accuracy": train_metrics["balanced_accuracy"],
                    "train_macro_f1": train_metrics["macro_f1"],
                    "val_accuracy": val_metrics["accuracy"],
                    "val_balanced_accuracy": val_metrics["balanced_accuracy"],
                    "val_macro_f1": val_metrics["macro_f1"],
                    "cnn_accuracy": test_metrics["accuracy"],
                    "cnn_balanced_accuracy": test_metrics["balanced_accuracy"],
                    "cnn_macro_f1": test_metrics["macro_f1"],
                    **representation_metrics,
                }
                rows.append(row)

                hist = history.copy()
                hist["view"] = view
                hist["group"] = group_name
                hist["sampling_rate_hz"] = rate
                hist["variant"] = variant
                hist["seed"] = master_seed
                histories.append(hist)

                ph = lp_history.copy()
                ph["view"] = view
                ph["group"] = group_name
                ph["sampling_rate_hz"] = rate
                ph["variant"] = variant
                ph["seed"] = master_seed
                probe_histories.append(ph)

                for sid, yt, yp in zip(sample_ids, y_true, y_pred, strict=True):
                    predictions.append({
                        "view": view,
                        "group": group_name,
                        "sampling_rate_hz": rate,
                        "variant": variant,
                        "seed": master_seed,
                        "sample_id": str(sid),
                        "y_true": int(yt),
                        "y_pred": int(yp),
                        "label_true": IDX_TO_CLASS[int(yt)],
                        "label_pred": IDX_TO_CLASS[int(yp)],
                    })

                if SAVE_CHECKPOINTS:
                    ckpt_dir = RESULTS_DIR / "checkpoints" / view / group_name / variant
                    ckpt_dir.mkdir(parents=True, exist_ok=True)
                    torch.save(
                        {
                            "model_state_dict": model.state_dict(),
                            "variant": variant,
                            "seed": master_seed,
                            "sampling_rate_hz": rate,
                            "view": view,
                            "class_to_idx": CLASS_TO_IDX,
                        },
                        ckpt_dir / f"seed_{master_seed}.pt",
                    )

    return (
        pd.DataFrame(rows),
        pd.concat(histories, ignore_index=True) if histories else pd.DataFrame(),
        pd.DataFrame(predictions),
        pd.concat(probe_histories, ignore_index=True) if probe_histories else pd.DataFrame(),
    )


RESULTS_DIR.mkdir(parents=True, exist_ok=True)
native_results, native_history, native_predictions, native_probe_history = run_reconstruction_sweep(
    "native", CNN_VARIANTS
)

native_results.to_csv(RESULTS_DIR / "experiment_1_0A_native_results.csv", index=False)
native_history.to_csv(RESULTS_DIR / "experiment_1_0A_native_training_history.csv", index=False)
native_predictions.to_csv(RESULTS_DIR / "experiment_1_0A_native_predictions.csv", index=False)
native_probe_history.to_csv(RESULTS_DIR / "experiment_1_0A_native_linear_probe_history.csv", index=False)
display(native_results)


## 1.0A summary tables and visualizations

The main plots use mean ± SD across the five paired seeds.

Interpretation hierarchy:

- If CNN-L stays stable across rates, there is no obvious practical downstream degradation.
- If linear-probe and kNN BA also stay stable, the learned representation remains comparably separable and locally class-structured.
- If only CNN-S degrades while CNN-L is stable, that is more consistent with a decoder-capacity limitation than information loss.
- Native-rate results must still be checked against 1.0B because the physical CNN receptive field changes with sample rate.


In [ ]:
def aggregate_results(df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "cnn_accuracy", "cnn_balanced_accuracy", "cnn_macro_f1",
        "linear_probe_balanced_accuracy", "linear_probe_macro_f1",
        "knn_balanced_accuracy", "knn_macro_f1",
    ]
    agg_spec = {}
    for metric in metric_cols:
        agg_spec[f"mean_{metric}"] = (metric, "mean")
        agg_spec[f"sd_{metric}"] = (metric, "std")
    return (
        df.groupby(["view", "sampling_rate_hz", "variant"], as_index=False)
        .agg(**agg_spec)
        .sort_values(["variant", "sampling_rate_hz"])
    )


def plot_sampling_rate_metric(df, metric, ylabel, title):
    summary = (
        df.groupby(["sampling_rate_hz", "variant"])[metric]
        .agg(["mean", "std"])
        .reset_index()
        .sort_values("sampling_rate_hz")
    )
    fig, ax = plt.subplots(figsize=(9.5, 5.6))
    for variant in CNN_VARIANTS:
        sub = summary[summary.variant == variant]
        if sub.empty:
            continue
        ax.errorbar(
            sub.sampling_rate_hz,
            sub["mean"],
            yerr=sub["std"],
            marker="o",
            capsize=4,
            label=variant.upper().replace("_", "-"),
        )
    ax.set_xlabel("Sampling rate (Hz)")
    ax.set_ylabel(ylabel)
    ax.set_title(title + "\nMean ± SD across 5 paired seeds")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()


native_summary = aggregate_results(native_results)
display(native_summary)

plot_sampling_rate_metric(
    native_results,
    "cnn_balanced_accuracy",
    "Test balanced accuracy",
    "Experiment 1.0A — Native-rate end-to-end CNN performance",
)
plot_sampling_rate_metric(
    native_results,
    "linear_probe_balanced_accuracy",
    "Linear-probe balanced accuracy",
    "Experiment 1.0A — Native-rate frozen-embedding linear separability",
)
plot_sampling_rate_metric(
    native_results,
    "knn_balanced_accuracy",
    "kNN balanced accuracy",
    "Experiment 1.0A — Native-rate embedding neighborhood quality",
)
plot_sampling_rate_metric(
    native_results,
    "cnn_macro_f1",
    "Test macro-F1",
    "Experiment 1.0A — Native-rate CNN macro-F1",
)


## 1.0A paired deltas relative to the highest-rate reference

Because the same five seeds are paired across rates, the most interpretable comparison is often the **within-seed delta**:

`delta(rate, seed) = metric(rate, seed) - metric(reference_rate, seed)`

The highest native rate is used as the reference. A horizontal line at `-PRACTICAL_BA_MARGIN` marks the configured practical degradation tolerance. This is an engineering threshold, not a formal equivalence test.


In [ ]:
def paired_delta_table(df: pd.DataFrame, metric: str, variant: str = "cnn_l") -> pd.DataFrame:
    ref_rate = float(max(df.sampling_rate_hz.unique()))
    sub = df[df.variant == variant].copy()
    ref = (
        sub[sub.sampling_rate_hz == ref_rate][["seed", metric]]
        .rename(columns={metric: "reference_value"})
    )
    merged = sub.merge(ref, on="seed", how="inner")
    merged["reference_rate_hz"] = ref_rate
    merged["delta"] = merged[metric] - merged["reference_value"]
    return merged


def plot_paired_delta(df, metric, ylabel, title, variant="cnn_l"):
    delta = paired_delta_table(df, metric, variant)
    summary = (
        delta.groupby("sampling_rate_hz")["delta"]
        .agg(["mean", "std"])
        .reset_index()
        .sort_values("sampling_rate_hz")
    )
    fig, ax = plt.subplots(figsize=(9.0, 5.2))
    ax.errorbar(summary.sampling_rate_hz, summary["mean"], yerr=summary["std"], marker="o", capsize=4)
    ax.axhline(0.0, linestyle="--", label="Reference performance")
    ax.axhline(-PRACTICAL_BA_MARGIN, linestyle=":", label=f"-{PRACTICAL_BA_MARGIN:.2f} practical margin")
    ax.set_xlabel("Sampling rate (Hz)")
    ax.set_ylabel(ylabel)
    ax.set_title(title + "\nPaired across 5 seeds")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()
    return delta, summary


native_ba_delta, native_ba_delta_summary = plot_paired_delta(
    native_results,
    "cnn_balanced_accuracy",
    "Delta balanced accuracy vs reference",
    "Experiment 1.0A — CNN-L paired BA change",
)
display(native_ba_delta_summary)

native_lp_delta, native_lp_delta_summary = plot_paired_delta(
    native_results,
    "linear_probe_balanced_accuracy",
    "Delta linear-probe BA vs reference",
    "Experiment 1.0A — CNN-L paired linear-probe change",
)


# Experiment 1.0B — Common-time-grid reconstruction control

This is the key control for the fact that CNN kernels are defined in samples.

All valid reconstructed signals are represented on `COMMON_GRID_RATE_HZ` before CNN training. Lower-rate signals are interpolated upward; this changes the numerical grid but does not reconstruct information that the lower-rate pipeline already removed.

After this transform, CNN-S/M/L have the **same physical receptive-field duration across all dataset groups**. A stable CNN-L result here is stronger evidence that lower-rate reconstruction preserved the task-relevant information rather than merely benefiting from a larger native-rate physical receptive field.


In [ ]:
if RUN_COMMON_GRID_CONTROL:
    common_results, common_history, common_predictions, common_probe_history = run_reconstruction_sweep(
        "common_grid", tuple(COMMON_GRID_VARIANTS)
    )
    common_results.to_csv(RESULTS_DIR / "experiment_1_0B_common_grid_results.csv", index=False)
    common_history.to_csv(RESULTS_DIR / "experiment_1_0B_common_grid_training_history.csv", index=False)
    common_predictions.to_csv(RESULTS_DIR / "experiment_1_0B_common_grid_predictions.csv", index=False)
    common_probe_history.to_csv(RESULTS_DIR / "experiment_1_0B_common_grid_linear_probe_history.csv", index=False)
    display(common_results)
else:
    common_results = pd.DataFrame()
    common_predictions = pd.DataFrame()
    print("Experiment 1.0B skipped by configuration.")


In [ ]:
if RUN_COMMON_GRID_CONTROL and not common_results.empty:
    common_summary = aggregate_results(common_results)
    display(common_summary)

    plot_sampling_rate_metric(
        common_results,
        "cnn_balanced_accuracy",
        "Test balanced accuracy",
        f"Experiment 1.0B — Common-grid CNN performance @ {COMMON_GRID_RATE_HZ:g} Hz",
    )
    plot_sampling_rate_metric(
        common_results,
        "linear_probe_balanced_accuracy",
        "Linear-probe balanced accuracy",
        f"Experiment 1.0B — Common-grid linear separability @ {COMMON_GRID_RATE_HZ:g} Hz",
    )
    plot_sampling_rate_metric(
        common_results,
        "knn_balanced_accuracy",
        "kNN balanced accuracy",
        f"Experiment 1.0B — Common-grid embedding neighborhoods @ {COMMON_GRID_RATE_HZ:g} Hz",
    )

    common_ba_delta, common_ba_delta_summary = plot_paired_delta(
        common_results,
        "cnn_balanced_accuracy",
        "Delta balanced accuracy vs reference",
        "Experiment 1.0B — CNN-L paired BA change on a common time grid",
    )
    display(common_ba_delta_summary)


# Experiment 1.0C — Direct weighted-event 10-bin control

This section reuses the strongest diagnostic from Experiment 1.3.

For each valid gesture:

1. Read the direct weighted event channels, excluding accel/gyro.
2. Split the valid prefix into 10 **relative-time** bins.
3. Sum each event channel inside each bin.
4. Flatten the `10 x C_event` matrix into one feature vector.
5. Standardize features using training statistics only.
6. Fit a multiclass linear classifier and evaluate on test.

The 10 bins encode *where in gesture progress* each event channel is active while discarding fine within-bin timing.

### Important schema rule

This comparison is meaningful only if the event representation is matched across sample rates. For example, 15 signed channels and 30 polarity-split channels are different representations and must not be interpreted as a pure sample-rate comparison. If schemas differ, this section skips by default.


In [ ]:
def group_event_schema(group_name: str):
    signature = group_signatures[group_name]
    return (
        signature["feature_schema"],
        int(signature["event_count"]),
        tuple(signature["event_names"]),
    )


def build_event_bin_features(group_name: str, split_name: str, n_bins: int):
    data = loaded_groups[group_name]
    df = common_manifests[group_name]
    df = df[df.split == split_name].sort_values("sample_id")
    event_count = int(group_signatures[group_name]["event_count"])

    features, labels, sample_ids = [], [], []
    for row in df.itertuples(index=False):
        p = data.packages[int(row.package_index)]
        i = int(row.segment_index)
        L = int(row.valid_length)
        if n_bins > L:
            raise ValueError(f"n_bins={n_bins} exceeds valid_length={L} for {row.sample_id}")
        x = np.asarray(p.padded_spike_imu[i, :L, :event_count], dtype=np.float64)
        chunks = np.array_split(x, n_bins, axis=0)
        feature = np.stack([chunk.sum(axis=0) for chunk in chunks], axis=0).reshape(-1)
        features.append(feature)
        labels.append(int(row.label_idx))
        sample_ids.append(str(row.sample_id))
    return np.asarray(features, dtype=np.float32), np.asarray(labels, dtype=np.int64), np.asarray(sample_ids)


def run_event_10bin_control():
    schemas = {name: group_event_schema(name) for name in loaded_groups}
    unique_schemas = {repr(v) for v in schemas.values()}
    if len(unique_schemas) != 1:
        message = "Event schemas are not identical across groups; 1.0C cannot isolate sample rate."
        print(message)
        display(pd.DataFrame([
            {
                "group": name,
                "sampling_rate_hz": group_rates[name],
                "feature_schema": schema[0],
                "event_count": schema[1],
                "event_names": schema[2],
            }
            for name, schema in schemas.items()
        ]))
        if SKIP_EVENT_CONTROL_ON_SCHEMA_MISMATCH:
            return pd.DataFrame()
        raise ValueError(message)

    rows = []
    for group_name in sorted(group_rates, key=group_rates.get, reverse=True):
        X_train, y_train, _ = build_event_bin_features(group_name, "train", EVENT_N_BINS)
        X_test, y_test, _ = build_event_bin_features(group_name, "test", EVENT_N_BINS)

        mean = X_train.mean(axis=0, keepdims=True)
        std = X_train.std(axis=0, keepdims=True)
        std[std < 1e-8] = 1.0
        X_train_z = (X_train - mean) / std
        X_test_z = (X_test - mean) / std

        for master_seed in SEEDS:
            probe_seed = derive_seed(master_seed, "event_10bin", "linear_probe")
            probe = LogisticRegression(
                max_iter=5000,
                random_state=probe_seed,
                solver="lbfgs",
            )
            probe.fit(X_train_z, y_train)
            y_pred = probe.predict(X_test_z)
            metrics = metric_dict(y_test, y_pred)
            rows.append({
                "group": group_name,
                "sampling_rate_hz": group_rates[group_name],
                "seed": master_seed,
                "n_bins": EVENT_N_BINS,
                "feature_dim": X_train.shape[1],
                "accuracy": metrics["accuracy"],
                "balanced_accuracy": metrics["balanced_accuracy"],
                "macro_f1": metrics["macro_f1"],
            })
    return pd.DataFrame(rows)


if RUN_EVENT_10BIN_CONTROL:
    event_results = run_event_10bin_control()
    if not event_results.empty:
        event_results.to_csv(RESULTS_DIR / "experiment_1_0C_event_10bin_results.csv", index=False)
        display(event_results)
else:
    event_results = pd.DataFrame()
    print("Experiment 1.0C skipped by configuration.")


In [ ]:
if RUN_EVENT_10BIN_CONTROL and not event_results.empty:
    event_summary = (
        event_results.groupby("sampling_rate_hz")
        .agg(
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            sd_balanced_accuracy=("balanced_accuracy", "std"),
            mean_accuracy=("accuracy", "mean"),
            mean_macro_f1=("macro_f1", "mean"),
            feature_dim=("feature_dim", "first"),
        )
        .reset_index()
        .sort_values("sampling_rate_hz")
    )
    display(event_summary)

    fig, ax = plt.subplots(figsize=(9.0, 5.2))
    ax.errorbar(
        event_summary.sampling_rate_hz,
        event_summary.mean_balanced_accuracy,
        yerr=event_summary.sd_balanced_accuracy,
        marker="o",
        capsize=4,
    )
    ax.set_xlabel("Sampling rate (Hz)")
    ax.set_ylabel("10-bin event linear-probe balanced accuracy")
    ax.set_title("Experiment 1.0C — Direct weighted-event temporal-position information\nMean ± SD across 5 seeds")
    ax.grid(True, alpha=0.25)
    plt.show()


## 10. CNN-L per-class recall change

Overall balanced accuracy can hide a class-specific failure. This diagnostic compares per-class recall for CNN-L at each native rate against the highest-rate reference using the paired test predictions.

A letter with a large negative delta may depend more strongly on fast motion components even when the overall mean is stable.


In [ ]:
def per_class_recall_from_predictions(predictions: pd.DataFrame, variant="cnn_l") -> pd.DataFrame:
    pred = predictions[predictions.variant == variant].copy()
    rows = []
    for (rate, seed), part in pred.groupby(["sampling_rate_hz", "seed"]):
        for class_index, class_name in IDX_TO_CLASS.items():
            selected = part.y_true == int(class_index)
            if not selected.any():
                continue
            recall = float((part.loc[selected, "y_pred"] == int(class_index)).mean())
            rows.append({
                "sampling_rate_hz": float(rate),
                "seed": int(seed),
                "class_index": int(class_index),
                "label": class_name,
                "recall": recall,
            })
    return pd.DataFrame(rows)


per_class_native = per_class_recall_from_predictions(native_predictions, "cnn_l")
ref_rate = float(max(group_rates.values()))
ref = per_class_native[per_class_native.sampling_rate_hz == ref_rate][["seed", "label", "recall"]].rename(
    columns={"recall": "reference_recall"}
)
per_class_delta = per_class_native.merge(ref, on=["seed", "label"], how="inner")
per_class_delta["delta_recall"] = per_class_delta.recall - per_class_delta.reference_recall

lowest_rate = float(min(group_rates.values()))
lowest_delta_summary = (
    per_class_delta[per_class_delta.sampling_rate_hz == lowest_rate]
    .groupby("label")["delta_recall"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean")
)
display(lowest_delta_summary)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.errorbar(
    np.arange(len(lowest_delta_summary)),
    lowest_delta_summary["mean"],
    yerr=lowest_delta_summary["std"],
    fmt="o",
    capsize=3,
)
ax.axhline(0.0, linestyle="--")
ax.set_xticks(np.arange(len(lowest_delta_summary)))
ax.set_xticklabels(lowest_delta_summary.label, rotation=45, ha="right")
ax.set_ylabel(f"Recall delta: {lowest_rate:g} Hz - {ref_rate:g} Hz")
ax.set_title("Experiment 1.0A — CNN-L per-class recall change")
ax.grid(True, axis="y", alpha=0.25)
plt.show()


## 11. Final evidence table

The table below gathers the quantities most directly relevant to the sampling-rate question. The highest-rate dataset is the reference.

Recommended interpretation:

- **Strong preservation evidence:** CNN-L native BA, CNN-L common-grid BA, linear-probe BA, and kNN BA all remain close to reference; event 10-bin BA is also stable when schemas match.
- **Decoder-dependent result:** native CNN performance is stable but common-grid performance changes substantially, suggesting that native physical receptive-field differences affected the comparison.
- **Representation degradation:** CNN-L, linear probe, and kNN all decrease consistently at lower rates, especially in the common-grid control.
- **Event-specific degradation:** reconstruction remains strong but event 10-bin performance drops, suggesting that reconstruction may be smoothing/compensating for event-domain losses.

Do not interpret `mean delta > -margin` as a formal statistical equivalence proof; it is an explicit practical engineering criterion.


In [ ]:
def cnn_l_evidence_rows(results: pd.DataFrame, view_label: str):
    if results.empty:
        return []
    sub = results[results.variant == "cnn_l"].copy()
    ref_rate = float(max(sub.sampling_rate_hz.unique()))
    metric_map = {
        "CNN-L test BA": "cnn_balanced_accuracy",
        "CNN-L linear-probe BA": "linear_probe_balanced_accuracy",
        "CNN-L kNN BA": "knn_balanced_accuracy",
        "CNN-L macro-F1": "cnn_macro_f1",
    }
    rows = []
    for rate in sorted(sub.sampling_rate_hz.unique()):
        for label, metric in metric_map.items():
            cur = sub[sub.sampling_rate_hz == rate][["seed", metric]]
            ref = sub[sub.sampling_rate_hz == ref_rate][["seed", metric]].rename(columns={metric: "ref"})
            paired = cur.merge(ref, on="seed", how="inner")
            delta = paired[metric] - paired["ref"]
            rows.append({
                "evidence": f"{view_label}: {label}",
                "sampling_rate_hz": float(rate),
                "reference_rate_hz": ref_rate,
                "mean_metric": float(paired[metric].mean()),
                "sd_metric": float(paired[metric].std(ddof=1)),
                "mean_paired_delta": float(delta.mean()),
                "sd_paired_delta": float(delta.std(ddof=1)),
                "within_3pt_like_margin": bool(delta.mean() >= -PRACTICAL_BA_MARGIN) if "BA" in label else np.nan,
            })
    return rows


evidence_rows = []
evidence_rows += cnn_l_evidence_rows(native_results, "native")
if RUN_COMMON_GRID_CONTROL and not common_results.empty:
    evidence_rows += cnn_l_evidence_rows(common_results, "common_grid")

if RUN_EVENT_10BIN_CONTROL and not event_results.empty:
    ref_rate = float(max(event_results.sampling_rate_hz.unique()))
    ref = event_results[event_results.sampling_rate_hz == ref_rate][["seed", "balanced_accuracy"]].rename(
        columns={"balanced_accuracy": "ref"}
    )
    for rate in sorted(event_results.sampling_rate_hz.unique()):
        cur = event_results[event_results.sampling_rate_hz == rate][["seed", "balanced_accuracy"]]
        paired = cur.merge(ref, on="seed", how="inner")
        delta = paired.balanced_accuracy - paired.ref
        evidence_rows.append({
            "evidence": f"event_10bin: linear-probe BA",
            "sampling_rate_hz": float(rate),
            "reference_rate_hz": ref_rate,
            "mean_metric": float(paired.balanced_accuracy.mean()),
            "sd_metric": float(paired.balanced_accuracy.std(ddof=1)),
            "mean_paired_delta": float(delta.mean()),
            "sd_paired_delta": float(delta.std(ddof=1)),
            "within_3pt_like_margin": bool(delta.mean() >= -PRACTICAL_BA_MARGIN),
        })

evidence_table = pd.DataFrame(evidence_rows)
evidence_table.to_csv(RESULTS_DIR / "experiment_1_0_final_evidence_table.csv", index=False)
display(evidence_table)


## 12. Review / audit checklist

Before using Experiment 1.0 to claim that a lower sample rate preserves classification information, verify all of the following:

1. **Matched acquisition/encoder setup:** sampling rate is the intended major changing variable. Wavelet frequencies, event polarity convention, thresholds, reconstruction implementation, labels, and segmentation policy are matched.
2. **Common cohort:** all reported rates use exactly the same canonical sample IDs after intersection.
3. **Label identity:** corresponding sample IDs have identical labels across rates.
4. **User-disjoint split:** train/validation/test users are disjoint and identical across rates.
5. **Class coverage:** every selected label appears in train, validation, and test.
6. **Paired seeds:** the same five master seeds are used at every rate; corresponding model initializations and DataLoader permutations are paired.
7. **Train-only normalization:** no validation/test statistics leak into normalization.
8. **Same training budget:** optimizer, learning rate, maximum epochs, patience, class weighting, and batch size are unchanged across rates.
9. **Validation-only selection:** best CNN epoch and kNN `k` are selected without test labels.
10. **Native RF audit:** do not interpret native-rate CNN differences without acknowledging that fixed kernels span different physical durations.
11. **Common-grid control:** use 1.0B as the stronger control when making an information-preservation statement.
12. **CNN-L priority:** CNN-S/M are capacity controls. A weak CNN-S is not evidence that the signal lacks information if CNN-L and representation probes remain strong.
13. **Event-schema match:** interpret 1.0C only when event channel definitions are matched across rates.
14. **Per-class check:** inspect whether a stable overall BA hides specific letters with large recall losses.
15. **Practical margin:** report paired deltas and the predefined degradation tolerance; avoid claiming formal equivalence from non-significance alone.

### Relationship to Experiments 1.1–1.3

- **1.0** asks whether changing sample rate changes the amount of usable classification information.
- **1.1** asks what temporal continuity is needed in reconstructed acceleration.
- **1.2** asks what temporal continuity the current direct-event CNN can exploit.
- **1.3** asks how much direct-event information is accessible from coarse relative temporal position alone.

If 1.0 shows that 64 Hz preserves reconstruction and event-domain classification information, later SNN design can use 64 Hz without treating the lower rate itself as the main information bottleneck.


## 13. Save run provenance

This lightweight provenance record is intended to make later review easier. It stores configuration and metadata summaries, not model tensors.


In [ ]:
provenance = {
    "experiment": "1.0_sampling_rate_information_preservation",
    "dataset_groups": {
        name: [str(Path(root)) for root in roots]
        for name, roots in DATASET_GROUPS.items()
    },
    "sampling_rates_hz": group_rates,
    "common_sample_count": len(common_ids),
    "class_to_idx": CLASS_TO_IDX,
    "seeds": list(SEEDS),
    "split_seed": SPLIT_SEED,
    "cnn_variants": list(CNN_VARIANTS),
    "training": {
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
    },
    "common_grid": {
        "enabled": RUN_COMMON_GRID_CONTROL,
        "rate_hz": COMMON_GRID_RATE_HZ,
        "variants": list(COMMON_GRID_VARIANTS),
        "pad_length": common_grid_pad_length,
    },
    "event_10bin": {
        "enabled": RUN_EVENT_10BIN_CONTROL,
        "n_bins": EVENT_N_BINS,
        "signatures": {name: repr(group_event_schema(name)) for name in loaded_groups},
    },
    "practical_ba_margin": PRACTICAL_BA_MARGIN,
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / "provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance, f, indent=2)
print("Saved artifacts to:", RESULTS_DIR)
